In [1]:
import pathlib
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split


def load_data(file_path="anime.csv"):

    # Read dataset
    df = pd.read_csv(file_path)

    expected_columns = [
        "anime_id",
        "name",
        "genre",
        "type",
        "episodes",
        "rating",
        "members"
    ]

    missing = [col for col in expected_columns if col not in df.columns]

    if missing:
        raise ValueError(f"Expected columns missing from anime.csv: {missing}")

    return df[expected_columns].copy()


def clean_data(df):
    df = df.copy()
    df["genre"] = df["genre"].fillna("").astype(str)
    df["type"] = df["type"].fillna("Unknown").astype(str)
    df["episodes"] = pd.to_numeric(df["episodes"], errors="coerce").fillna(0)
    df["rating"] = pd.to_numeric(df["rating"], errors="coerce")
    df["members"] = pd.to_numeric(df["members"], errors="coerce")
    df = df.assign(
        rating=df["rating"].fillna(df["rating"].mean()),
        members=df["members"].fillna(df["members"].mean()),
    )
    return df


def fit_transformers(df):
    df = clean_data(df)
    genre_text = df["genre"].str.replace(",", " ", regex=False)

    genre_vectorizer = CountVectorizer(token_pattern=r"(?u)\b\w+\b")
    genre_vectorizer.fit(genre_text)

    try:
        type_encoder = OneHotEncoder(sparse=False, handle_unknown="ignore")
    except TypeError:
        type_encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
    type_encoder.fit(df[["type"]])

    scaler = StandardScaler()
    scaler.fit(df[["episodes", "rating", "members"]])

    return genre_vectorizer, type_encoder, scaler


def transform_features(df, genre_vectorizer, type_encoder, scaler):
    df = clean_data(df)
    genre_text = df["genre"].str.replace(",", " ", regex=False)
    genre_matrix = genre_vectorizer.transform(genre_text).toarray()
    type_matrix = type_encoder.transform(df[["type"]])
    numeric_matrix = scaler.transform(df[["episodes", "rating", "members"]])
    features = np.hstack([genre_matrix, type_matrix, numeric_matrix])
    return features, df


def recommend_anime(title, data, features, top_n=10, similarity_threshold=0.0):
    title_lower = title.strip().lower()
    matched = data[data["name"].str.lower() == title_lower]
    if matched.empty:
        raise ValueError(f"Anime title not found: {title}")

    query_index = matched.index[0]
    query_vector = features[query_index].reshape(1, -1)
    scores = cosine_similarity(query_vector, features)[0]
    scores[query_index] = -1

    if similarity_threshold > 0:
        eligible_indices = np.where(scores >= similarity_threshold)[0]
        eligible_indices = eligible_indices[eligible_indices != query_index]
        if len(eligible_indices) == 0:
            eligible_indices = np.where(scores != -1)[0]
        top_indices = eligible_indices[np.argsort(scores[eligible_indices])[::-1][:top_n]]
    else:
        top_indices = np.argsort(scores)[::-1][:top_n]

    recommendations = data.iloc[top_indices].reset_index(drop=True)
    recommendations["similarity_score"] = scores[top_indices]
    return recommendations


def build_train_test(data, test_size=0.2, random_state=42):
    train_df, test_df = train_test_split(data, test_size=test_size, random_state=random_state)
    train_df = train_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)
    return train_df, test_df


def genre_tokens(genre_value):
    tokens = [token.strip().lower() for token in str(genre_value).split(",") if token.strip()]
    return set(tokens)


def evaluate_recommendations(train_df, train_features, test_df, test_features, top_n=10):
    train_genre_sets = [genre_tokens(v) for v in train_df["genre"]]
    precision_scores = []
    recall_scores = []
    f1_scores = []

    for i, test_row in test_df.iterrows():
        target_genres = genre_tokens(test_row["genre"])
        if not target_genres:
            continue

        similarity_row = cosine_similarity(test_features[i].reshape(1, -1), train_features)[0]
        recommended_indices = np.argsort(similarity_row)[::-1][:top_n]

        relevant_indices = [j for j, genres in enumerate(train_genre_sets) if genres & target_genres]
        true_positives = len(set(recommended_indices) & set(relevant_indices))

        precision = true_positives / top_n if top_n else 0.0
        recall = true_positives / len(relevant_indices) if relevant_indices else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

        precision_scores.append(precision)
        recall_scores.append(recall)
        f1_scores.append(f1)

    if not precision_scores:
        return 0.0, 0.0, 0.0

    return (
        float(np.mean(precision_scores)),
        float(np.mean(recall_scores)),
        float(np.mean(f1_scores)),
    )


def main():
    data = load_data()
    print("Loaded anime dataset with", len(data), "rows")

    data = clean_data(data)
    train_df, test_df = build_train_test(data)

    transformers = fit_transformers(train_df)
    train_features, train_df = transform_features(train_df, *transformers)
    test_features, test_df = transform_features(test_df, *transformers)
    full_features, full_data = transform_features(data, *transformers)

    precision, recall, f1 = evaluate_recommendations(train_df, train_features, test_df, test_features)
    print("\nEvaluation Results")
    print("Precision:", round(precision, 4))
    print("Recall:", round(recall, 4))
    print("F1 Score:", round(f1, 4))

    example_title = data.loc[0, "name"]
    print(f"\nRecommendations for '{example_title}':")

    thresholds = [0.0, 0.1, 0.2, 0.3]
    for threshold in thresholds:
        recommendations = recommend_anime(
            example_title,
            full_data,
            full_features,
            top_n=10,
            similarity_threshold=threshold,
        )
        print(f"\nThreshold {threshold:.1f}: {len(recommendations)} recommendations")
        print(recommendations[["anime_id", "name", "genre", "type", "episodes", "rating", "similarity_score"]].to_string(index=False))


if __name__ == "__main__":
    main()


Loaded anime dataset with 12294 rows

Evaluation Results
Precision: 0.9903
Recall: 0.0049
F1 Score: 0.0097

Recommendations for 'Kimi no Na wa.':

Threshold 0.0: 10 recommendations
 anime_id                                                     name                                                  genre    type  episodes  rating  similarity_score
    10408                                       Hotarubi no Mori e                   Drama, Romance, Shoujo, Supernatural   Movie       1.0    8.61          0.945877
      578                                           Hotaru no Haka                                      Drama, Historical   Movie       1.0    8.58          0.899768
     7311                            Suzumiya Haruhi no Shoushitsu Comedy, Mystery, Romance, School, Sci-Fi, Supernatural   Movie       1.0    8.81          0.895854
     6351    Clannad: After Story - Mou Hitotsu no Sekai, Kyou-hen                                 Drama, Romance, School Special       1.0    8.02        